# NBA Game Outcome Prediction - Data Processing

## Data Sources

This project uses NBA game data from the NBA Stats API for the 2024-25 and 2025-26 NBA regular seasons. The raw game data contains team-level box score statistics, including points, rebounds, assists, field goal percentage, three-point percentage, turnovers, and game results.

Betting market data was collected from a separate NBA closing-line dataset covering the same two seasons. The closing home point spread (`close_spread_home`) is retained for later comparison between machine learning predictions and the betting market.

The goal of this notebook is to clean and merge the two data sources and construct pre-game rolling features for model training.

## Data Sources
NBA game data from the 2024-25 and 2025-26 seasons were loaded and combined into a single dataset. The NBA Stats API provides one row for each team in each game, so each NBA game initially appears as two rows.

I first verified that each game contained two team records and examined gthe home/away designation in the 'MATCHUP' collum. Games without a standard home ('vs.') and away ('@) designation were excluded because home-curt status is important for this project and asigning a home team arbitrarily coud introduce incorrect information.

In [393]:
import pandas as pd

df_2024 = pd.read_excel("nba_2024_25_raw.xlsx")
df_2025 = pd.read_excel("nba_2025_26_raw.xlsx")

print("2024-25 shape:", df_2024.shape)
print("2025-26 shape:", df_2025.shape)

2024-25 shape: (2460, 29)
2025-26 shape: (2460, 29)


In [394]:
df_2024.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22024,1610612738,BOS,Boston Celtics,22400061,2024-10-22,BOS vs. NYK,W,240,48,...,29,40,33,6,3,4,15,132,23,1
1,22024,1610612750,MIN,Minnesota Timberwolves,22400062,2024-10-22,MIN @ LAL,L,240,35,...,35,47,17,4,1,16,22,103,-7,1
2,22024,1610612747,LAL,Los Angeles Lakers,22400062,2024-10-22,LAL vs. MIN,W,240,42,...,31,46,22,7,8,7,22,110,7,1
3,22024,1610612752,NYK,New York Knicks,22400061,2024-10-22,NYK @ BOS,L,240,43,...,29,34,20,2,3,12,12,109,-23,1
4,22024,1610612744,GSW,Golden State Warriors,22400072,2024-10-23,GSW @ POR,W,240,48,...,42,57,38,13,5,18,27,140,36,1


In [395]:
df_2024.columns.tolist()

['SEASON_ID',
 'TEAM_ID',
 'TEAM_ABBREVIATION',
 'TEAM_NAME',
 'GAME_ID',
 'GAME_DATE',
 'MATCHUP',
 'WL',
 'MIN',
 'FGM',
 'FGA',
 'FG_PCT',
 'FG3M',
 'FG3A',
 'FG3_PCT',
 'FTM',
 'FTA',
 'FT_PCT',
 'OREB',
 'DREB',
 'REB',
 'AST',
 'STL',
 'BLK',
 'TOV',
 'PF',
 'PTS',
 'PLUS_MINUS',
 'VIDEO_AVAILABLE']

In [396]:
### combine two season as a new column "season"
df_2024["SEASON"] = "2024-25"
df_2025["SEASON"] = "2025-26"

df = pd.concat([df_2024, df_2025], ignore_index=True)

print("Combined shape:", df.shape)
print(df["SEASON"].value_counts())

Combined shape: (4920, 30)
SEASON
2024-25    2460
2025-26    2460
Name: count, dtype: int64


In [397]:
print("Total rows:", len(df))
print("Unique games:", df["GAME_ID"].nunique())
print("Rows per game:")
print(df.groupby("GAME_ID").size().value_counts())

Total rows: 4920
Unique games: 2460
Rows per game:
2    2460
Name: count, dtype: int64


In [398]:
home_df = df[df["MATCHUP"].str.contains("vs.")].copy()
away_df = df[df["MATCHUP"].str.contains("@")].copy()

print("Home rows:", len(home_df))
print("Away rows:", len(away_df))

Home rows: 2450
Away rows: 2470


In [399]:
# Check games where home/away identification is unusual
home_count = df.groupby("GAME_ID")["MATCHUP"].apply(
    lambda x: x.str.contains("vs.", regex=False).sum()
)

unusual_games = home_count[home_count != 1]

print("Number of unusual games:", len(unusual_games))
print(unusual_games.head(20))

Number of unusual games: 10
GAME_ID
22400147    0
22400621    0
22400633    0
22401229    0
22401230    0
22500147    0
22500578    0
22500602    0
22501229    0
22501230    0
Name: MATCHUP, dtype: int64


In [400]:
df[df["GAME_ID"].isin(unusual_games.index)][
    [
        "SEASON",
        "GAME_ID",
        "GAME_DATE",
        "TEAM_ABBREVIATION",
        "MATCHUP",
        "WL"
    ]
].sort_values(["SEASON", "GAME_DATE", "GAME_ID"])

,SEASON,GAME_ID,GAME_DATE,TEAM_ABBREVIATION,MATCHUP,WL
162,2024-25,22400147,2024-11-02,MIA,MIA @ WAS,W
167,2024-25,22400147,2024-11-02,WAS,WAS @ MIA,L
748,2024-25,22401229,2024-12-14,ATL,ATL @ MIL,L
750,2024-25,22401229,2024-12-14,MIL,MIL @ ATL,W
749,2024-25,22401230,2024-12-14,HOU,HOU @ OKC,L
751,2024-25,22401230,2024-12-14,OKC,OKC @ HOU,W
1298,2024-25,22400621,2025-01-23,SAS,SAS @ IND,W
1305,2024-25,22400621,2025-01-23,IND,IND @ SAS,L
1321,2024-25,22400633,2025-01-25,IND,IND @ SAS,W
1336,2024-25,22400633,2025-01-25,SAS,SAS @ IND,L


In [401]:
### delete the unusual game which without home/away designation
df_clean = df[~df["GAME_ID"].isin(unusual_games.index)].copy()

print("Rows after removing unusual games:", len(df_clean))
print("Games after removing unusual games:", df_clean["GAME_ID"].nunique())

Rows after removing unusual games: 4900
Games after removing unusual games: 2450


The NBA stats API provide seperate rows fro the home and away teams in each game. I merged the two team-level rows using  'GAME_ID' and 'SEASON' to create on row pergame. After merging, each wor contains statistic for both the home and away teams and I also create the 'HOME_WIN' label. Using '1' for Home team win and '0' for home tem lost

In [402]:
home_df = df_clean[
    df_clean["MATCHUP"].str.contains("vs.", regex=False)
].copy()

away_df = df_clean[
    df_clean["MATCHUP"].str.contains("@", regex=False)
].copy()

print("Home rows:", len(home_df))
print("Away rows:", len(away_df))

Home rows: 2450
Away rows: 2450


In [403]:
game_df = home_df.merge(
    away_df,
    on=["GAME_ID", "SEASON"],
    suffixes=("_HOME", "_AWAY")
)

print("Game-level shape:", game_df.shape)
print("Number of games:", len(game_df))

Game-level shape: (2450, 58)
Number of games: 2450


In [404]:
games = game_df[[
    "SEASON",
    "GAME_ID",
    "GAME_DATE_HOME",

    "TEAM_ABBREVIATION_HOME",
    "TEAM_ABBREVIATION_AWAY",

    "WL_HOME",

    "PTS_HOME",
    "PTS_AWAY",

    "REB_HOME",
    "REB_AWAY",

    "AST_HOME",
    "AST_AWAY",

    "FG_PCT_HOME",
    "FG_PCT_AWAY",

    "FG3_PCT_HOME",
    "FG3_PCT_AWAY",

    "TOV_HOME",
    "TOV_AWAY",

    "PLUS_MINUS_HOME",
    "PLUS_MINUS_AWAY"
]].copy()

In [405]:
games = games.rename(columns={
    "GAME_DATE_HOME": "GAME_DATE",
    "TEAM_ABBREVIATION_HOME": "HOME_TEAM",
    "TEAM_ABBREVIATION_AWAY": "AWAY_TEAM"
})

In [406]:
games["HOME_WIN"] = (games["WL_HOME"] == "W").astype(int)

In [407]:
games.head()

,SEASON,GAME_ID,GAME_DATE,HOME_TEAM,AWAY_TEAM,WL_HOME,PTS_HOME,PTS_AWAY,REB_HOME,REB_AWAY,...,AST_AWAY,FG_PCT_HOME,FG_PCT_AWAY,FG3_PCT_HOME,FG3_PCT_AWAY,TOV_HOME,TOV_AWAY,PLUS_MINUS_HOME,PLUS_MINUS_AWAY,HOME_WIN
0,2024-25,22400061,2024-10-22,BOS,NYK,W,132,109,40,34,...,20,0.505,0.551,0.475,0.367,4,12,23,-23,1
1,2024-25,22400062,2024-10-22,LAL,MIN,W,110,103,46,47,...,17,0.442,0.412,0.167,0.317,7,16,7,-7,1
2,2024-25,22400068,2024-10-23,HOU,CHA,L,105,110,43,56,...,20,0.369,0.447,0.302,0.385,8,17,-5,5,0
3,2024-25,22400071,2024-10-23,LAC,PHX,L,113,116,51,42,...,25,0.457,0.487,0.276,0.395,22,22,-3,3,0
4,2024-25,22400072,2024-10-23,POR,GSW,L,104,140,42,57,...,38,0.391,0.516,0.235,0.417,18,18,-36,36,0


In order to represent each team's recent performance before a game, I created rolling features based on the team's previous 5 and previous 10 game. The important to feature for the machine learning to predict the game winner include points (PTS), rebounds (REB), assists (AST), field goal percentage (FG_PCT), three-point percentage (FG3_PCT), turnovers (TOV), and win rate (WIN).

In [408]:
games["GAME_DATE"] = pd.to_datetime(games["GAME_DATE"])

games = games.sort_values(["GAME_DATE", "GAME_ID"]).reset_index(drop=True)

games[["GAME_DATE", "HOME_TEAM", "AWAY_TEAM", "HOME_WIN"]].head(10)

,GAME_DATE,HOME_TEAM,AWAY_TEAM,HOME_WIN
0,2024-10-22,BOS,NYK,1
1,2024-10-22,LAL,MIN,1
2,2024-10-23,DET,IND,0
3,2024-10-23,ATL,BKN,1
4,2024-10-23,MIA,ORL,0
5,2024-10-23,PHI,MIL,0
6,2024-10-23,TOR,CLE,0
7,2024-10-23,HOU,CHA,0
8,2024-10-23,NOP,CHI,1
9,2024-10-23,UTA,MEM,0


In [409]:
home_history = games[[
    "SEASON", "GAME_ID", "GAME_DATE", "HOME_TEAM",
    "PTS_HOME", "REB_HOME", "AST_HOME",
    "FG_PCT_HOME", "FG3_PCT_HOME", "TOV_HOME",
    "PLUS_MINUS_HOME",
    "HOME_WIN"
]].copy()

home_history.columns = [
    "SEASON", "GAME_ID", "GAME_DATE", "TEAM",
    "PTS", "REB", "AST",
    "FG_PCT", "FG3_PCT", "TOV",
    "PLUS_MINUS",
    "WIN"
]

away_history = games[[
    "SEASON", "GAME_ID", "GAME_DATE", "AWAY_TEAM",
    "PTS_AWAY", "REB_AWAY", "AST_AWAY",
    "FG_PCT_AWAY", "FG3_PCT_AWAY", "TOV_AWAY",
    "PLUS_MINUS_AWAY",
    "HOME_WIN"
]].copy()

away_history.columns = [
    "SEASON", "GAME_ID", "GAME_DATE", "TEAM",
    "PTS", "REB", "AST",
    "FG_PCT", "FG3_PCT", "TOV",
    "PLUS_MINUS",
    "WIN"
]

away_history["WIN"] = 1 - away_history["WIN"]

In [410]:
team_history = pd.concat(
    [home_history, away_history],
    ignore_index=True
)

team_history = team_history.sort_values(
    ["SEASON", "TEAM", "GAME_DATE", "GAME_ID"]
).reset_index(drop=True)

team_history.head(20)

,SEASON,GAME_ID,GAME_DATE,TEAM,PTS,REB,AST,FG_PCT,FG3_PCT,TOV,PLUS_MINUS,WIN
0,2024-25,22400064,2024-10-23,ATL,120,45,25,0.488,0.321,16,4,1
1,2024-25,22400079,2024-10-25,ATL,125,39,25,0.481,0.368,14,5,1
2,2024-25,22400100,2024-10-27,ATL,104,49,24,0.396,0.323,20,-24,0
3,2024-25,22400103,2024-10-28,ATL,119,39,32,0.481,0.375,16,-2,0
4,2024-25,22400121,2024-10-30,ATL,120,41,28,0.474,0.308,17,-13,0
5,2024-25,22400135,2024-11-01,ATL,115,40,30,0.446,0.380,13,-8,0
6,2024-25,22400152,2024-11-03,ATL,126,41,29,0.568,0.417,16,15,1
7,2024-25,22400157,2024-11-04,ATL,93,43,23,0.416,0.194,20,-30,0
8,2024-25,22400171,2024-11-06,ATL,121,48,31,0.450,0.395,8,5,1
9,2024-25,22400185,2024-11-08,ATL,121,44,30,0.454,0.417,18,-1,0


In [411]:
##check the form status
team_history[
    (team_history["SEASON"] == "2024-25") &
    (team_history["TEAM"] == "BOS")
].head(10)

,SEASON,GAME_ID,GAME_DATE,TEAM,PTS,REB,AST,FG_PCT,FG3_PCT,TOV,PLUS_MINUS,WIN
163,2024-25,22400061,2024-10-22,BOS,132,40,33,0.505,0.475,4,23,1
164,2024-25,22400073,2024-10-24,BOS,122,49,21,0.467,0.378,14,20,1
165,2024-25,22400089,2024-10-26,BOS,124,37,19,0.422,0.458,10,6,1
166,2024-25,22400104,2024-10-28,BOS,119,38,24,0.494,0.383,14,11,1
167,2024-25,22400119,2024-10-30,BOS,132,51,23,0.404,0.333,14,-3,0
168,2024-25,22400132,2024-11-01,BOS,124,41,21,0.477,0.310,10,15,1
169,2024-25,22400141,2024-11-02,BOS,113,44,23,0.439,0.308,15,10,1
170,2024-25,22400157,2024-11-04,BOS,123,50,27,0.485,0.327,13,30,1
171,2024-25,22400172,2024-11-06,BOS,112,47,22,0.422,0.352,12,-6,0
172,2024-25,22400187,2024-11-08,BOS,108,39,23,0.433,0.264,13,4,1


Last 5 feature

In [412]:

stats = [
    "PTS",
    "REB",
    "AST",
    "FG_PCT",
    "FG3_PCT",
    "TOV",
    "PLUS_MINUS",
    "WIN"
]

for stat in stats:
    team_history[f"{stat}_LAST5"] = (
        team_history
        .groupby(["SEASON", "TEAM"])[stat]
        .transform(lambda x: x.shift(1).rolling(5).mean())   ###shift 1 means we cannot use the current game data
    )

In [413]:
team_history[
    (team_history["SEASON"] == "2024-25") &
    (team_history["TEAM"] == "BOS")
][[
    "GAME_DATE",
    "TEAM",
    "PTS",
    "PTS_LAST5",
    "REB_LAST5",
    "AST_LAST5",
    "WIN_LAST5"
]].head(10)

,GAME_DATE,TEAM,PTS,PTS_LAST5,REB_LAST5,AST_LAST5,WIN_LAST5
163,2024-10-22,BOS,132,NaN,NaN,NaN,NaN
164,2024-10-24,BOS,122,NaN,NaN,NaN,NaN
165,2024-10-26,BOS,124,NaN,NaN,NaN,NaN
166,2024-10-28,BOS,119,NaN,NaN,NaN,NaN
167,2024-10-30,BOS,132,NaN,NaN,NaN,NaN
168,2024-11-01,BOS,124,125.8,43.0,24.0,0.8
169,2024-11-02,BOS,113,124.2,43.2,21.6,0.8
170,2024-11-04,BOS,123,122.4,42.2,22.0,0.8
171,2024-11-06,BOS,112,122.2,44.8,23.6,0.8
172,2024-11-08,BOS,108,120.8,46.6,23.2,0.6


Last 10 features

In [414]:
for stat in stats:
    team_history[f"{stat}_LAST10"] = (
        team_history
        .groupby(["SEASON", "TEAM"])[stat]
        .transform(lambda x: x.shift(1).rolling(10).mean())
    )

team_history[
    (team_history["SEASON"] == "2024-25") &
    (team_history["TEAM"] == "BOS")
][[
    "GAME_DATE",
    "TEAM",
    "PTS",
    "PTS_LAST5",
    "PTS_LAST10",
    "WIN_LAST5",
    "WIN_LAST10"
]].head(15)

,GAME_DATE,TEAM,PTS,PTS_LAST5,PTS_LAST10,WIN_LAST5,WIN_LAST10
163,2024-10-22,BOS,132,NaN,NaN,NaN,NaN
164,2024-10-24,BOS,122,NaN,NaN,NaN,NaN
165,2024-10-26,BOS,124,NaN,NaN,NaN,NaN
166,2024-10-28,BOS,119,NaN,NaN,NaN,NaN
167,2024-10-30,BOS,132,NaN,NaN,NaN,NaN
168,2024-11-01,BOS,124,125.8,NaN,0.8,NaN
169,2024-11-02,BOS,113,124.2,NaN,0.8,NaN
170,2024-11-04,BOS,123,122.4,NaN,0.8,NaN
171,2024-11-06,BOS,112,122.2,NaN,0.8,NaN
172,2024-11-08,BOS,108,120.8,NaN,0.6,NaN


In [415]:
rolling_cols = []

for stat in stats:
    rolling_cols.append(f"{stat}_LAST5")
    rolling_cols.append(f"{stat}_LAST10")

rolling_cols

['PTS_LAST5',
 'PTS_LAST10',
 'REB_LAST5',
 'REB_LAST10',
 'AST_LAST5',
 'AST_LAST10',
 'FG_PCT_LAST5',
 'FG_PCT_LAST10',
 'FG3_PCT_LAST5',
 'FG3_PCT_LAST10',
 'TOV_LAST5',
 'TOV_LAST10',
 'PLUS_MINUS_LAST5',
 'PLUS_MINUS_LAST10',
 'WIN_LAST5',
 'WIN_LAST10']

In [416]:
home_features = team_history[
    ["SEASON", "GAME_ID", "TEAM"] + rolling_cols
].copy()

home_features = home_features.rename(
    columns={
        "TEAM": "HOME_TEAM",
        **{col: f"HOME_{col}" for col in rolling_cols}
    }
)

away_features = team_history[
    ["SEASON", "GAME_ID", "TEAM"] + rolling_cols
].copy()

away_features = away_features.rename(
    columns={
        "TEAM": "AWAY_TEAM",
        **{col: f"AWAY_{col}" for col in rolling_cols}
    }
)

In [417]:
model_data = games.merge(
    home_features,
    on=["SEASON", "GAME_ID", "HOME_TEAM"],
    how="left"
)

model_data = model_data.merge(
    away_features,
    on=["SEASON", "GAME_ID", "AWAY_TEAM"],
    how="left"
)

print("Shape:", model_data.shape)
model_data[[
    "GAME_DATE",
    "HOME_TEAM",
    "AWAY_TEAM",
    "HOME_PTS_LAST5",
    "AWAY_PTS_LAST5",
    "HOME_WIN_LAST5",
    "AWAY_WIN_LAST5",
    "HOME_PTS_LAST10",
    "AWAY_PTS_LAST10",
    "HOME_WIN"
]].head(20)

Shape: (2450, 53)


,GAME_DATE,HOME_TEAM,AWAY_TEAM,HOME_PTS_LAST5,AWAY_PTS_LAST5,HOME_WIN_LAST5,AWAY_WIN_LAST5,HOME_PTS_LAST10,AWAY_PTS_LAST10,HOME_WIN
0,2024-10-22,BOS,NYK,NaN,NaN,NaN,NaN,NaN,NaN,1
1,2024-10-22,LAL,MIN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,2024-10-23,DET,IND,NaN,NaN,NaN,NaN,NaN,NaN,0
3,2024-10-23,ATL,BKN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2024-10-23,MIA,ORL,NaN,NaN,NaN,NaN,NaN,NaN,0
5,2024-10-23,PHI,MIL,NaN,NaN,NaN,NaN,NaN,NaN,0
6,2024-10-23,TOR,CLE,NaN,NaN,NaN,NaN,NaN,NaN,0
7,2024-10-23,HOU,CHA,NaN,NaN,NaN,NaN,NaN,NaN,0
8,2024-10-23,NOP,CHI,NaN,NaN,NaN,NaN,NaN,NaN,1
9,2024-10-23,UTA,MEM,NaN,NaN,NaN,NaN,NaN,NaN,0


Load nba_2024_25_closing_line.cvs and nba_2025_26_closing_line.csv

For this project, I retain the closing home point spread ('close_spread_home') as the betting market benchmark. The closing spread represents the market's final assessment of the expected difference between the home and away teams before the game begins.   

In [418]:
odds_2024 = pd.read_csv("nba_2024_25_closing_line.csv")
odds_2025 = pd.read_csv("nba_2025_26_closing_line.csv")

print("2024-25 odds shape:", odds_2024.shape)
print("2025-26 odds shape:", odds_2025.shape)

print("\n2024-25 columns:")
print(odds_2024.columns.tolist())

2024-25 odds shape: (1314, 34)
2025-26 odds shape: (1315, 34)

2024-25 columns:
['game_id', 'date_et', 'start_time_utc', 'season', 'season_type', 'stage', 'round', 'home_team', 'away_team', 'home_team_id', 'away_team_id', 'home_score', 'away_score', 'overtime', 'bookmakers', 'open_home_win_prob', 'close_home_win_prob', 'close_fair_home_odds', 'close_fair_away_odds', 'open_spread_home', 'close_spread_home', 'close_home_cover_prob', 'open_total', 'close_total', 'close_over_prob', 'moneyline_winner', 'favorite', 'favorite_won', 'ats_result', 'ats_margin', 'ats_result_open', 'total_result', 'total_margin', 'total_result_open']


In [419]:
odds_2024 = odds_2024[
    odds_2024["season_type"] == "regular"
].copy()

odds_2025 = odds_2025[
    odds_2025["season_type"] == "regular"
].copy()

print("2024-25 regular season:", len(odds_2024))
print("2025-26 regular season:", len(odds_2025))

2024-25 regular season: 1224
2025-26 regular season: 1224


In [420]:
odds = pd.concat(
    [odds_2024, odds_2025],
    ignore_index=True
)

odds["date_et"] = pd.to_datetime(odds["date_et"])

print("Combined odds shape:", odds.shape)

odds[[
    "season",
    "date_et",
    "home_team",
    "away_team"
]].head(10)

Combined odds shape: (2448, 34)


,season,date_et,home_team,away_team
0,2024-25,2024-10-22,Boston Celtics,New York Knicks
1,2024-25,2024-10-22,Los Angeles Lakers,Minnesota Timberwolves
2,2024-25,2024-10-23,Detroit Pistons,Indiana Pacers
3,2024-25,2024-10-23,Philadelphia 76ers,Milwaukee Bucks
4,2024-25,2024-10-23,Toronto Raptors,Cleveland Cavaliers
5,2024-25,2024-10-23,Miami Heat,Orlando Magic
6,2024-25,2024-10-23,Atlanta Hawks,Brooklyn Nets
7,2024-25,2024-10-23,Houston Rockets,Charlotte Hornets
8,2024-25,2024-10-23,New Orleans Pelicans,Chicago Bulls
9,2024-25,2024-10-23,Utah Jazz,Memphis Grizzlies


In [421]:
team_mapping = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BKN",
    "Charlotte Hornets": "CHA",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHX",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

odds["HOME_TEAM"] = odds["home_team"].map(team_mapping)
odds["AWAY_TEAM"] = odds["away_team"].map(team_mapping)

In [422]:
print("Missing home teams:", odds["HOME_TEAM"].isna().sum())
print("Missing away teams:", odds["AWAY_TEAM"].isna().sum())

print("\nUnmatched home team names:")
print(odds.loc[odds["HOME_TEAM"].isna(), "home_team"].unique())

print("\nUnmatched away team names:")
print(odds.loc[odds["AWAY_TEAM"].isna(), "away_team"].unique())

Missing home teams: 0
Missing away teams: 0

Unmatched home team names:
[]

Unmatched away team names:
[]


In [423]:
odds[[
    "season",
    "date_et",
    "home_team",
    "HOME_TEAM",
    "away_team",
    "AWAY_TEAM"
]].head(10)

,season,date_et,home_team,HOME_TEAM,away_team,AWAY_TEAM
0,2024-25,2024-10-22,Boston Celtics,BOS,New York Knicks,NYK
1,2024-25,2024-10-22,Los Angeles Lakers,LAL,Minnesota Timberwolves,MIN
2,2024-25,2024-10-23,Detroit Pistons,DET,Indiana Pacers,IND
3,2024-25,2024-10-23,Philadelphia 76ers,PHI,Milwaukee Bucks,MIL
4,2024-25,2024-10-23,Toronto Raptors,TOR,Cleveland Cavaliers,CLE
5,2024-25,2024-10-23,Miami Heat,MIA,Orlando Magic,ORL
6,2024-25,2024-10-23,Atlanta Hawks,ATL,Brooklyn Nets,BKN
7,2024-25,2024-10-23,Houston Rockets,HOU,Charlotte Hornets,CHA
8,2024-25,2024-10-23,New Orleans Pelicans,NOP,Chicago Bulls,CHI
9,2024-25,2024-10-23,Utah Jazz,UTA,Memphis Grizzlies,MEM


In [424]:
odds["GAME_DATE"] = pd.to_datetime(odds["date_et"])
odds["SEASON"] = odds["season"]

print(odds[[
    "SEASON",
    "GAME_DATE",
    "HOME_TEAM",
    "AWAY_TEAM"
]].head())


    SEASON  GAME_DATE HOME_TEAM AWAY_TEAM
0  2024-25 2024-10-22       BOS       NYK
1  2024-25 2024-10-22       LAL       MIN
2  2024-25 2024-10-23       DET       IND
3  2024-25 2024-10-23       PHI       MIL
4  2024-25 2024-10-23       TOR       CLE


2442 games were successfull

In [425]:
merge_test = model_data.merge(
    odds,
    on=[
        "SEASON",
        "GAME_DATE",
        "HOME_TEAM",
        "AWAY_TEAM"
    ],
    how="left",
    indicator=True
)

print(merge_test["_merge"].value_counts())

_merge
both          2442
left_only        8
right_only       0
Name: count, dtype: int64


In [426]:
####find the left game
unmatched = merge_test[
    merge_test["_merge"] == "left_only"
][[
    "SEASON",
    "GAME_DATE",
    "HOME_TEAM",
    "AWAY_TEAM"
]]

unmatched


,SEASON,GAME_DATE,HOME_TEAM,AWAY_TEAM
358,2024-25,2024-12-10,MIL,ORL
359,2024-25,2024-12-10,OKC,DAL
360,2024-25,2024-12-11,NYK,ATL
361,2024-25,2024-12-11,HOU,GSW
1581,2025-26,2025-12-09,ORL,MIA
1582,2025-26,2025-12-09,TOR,NYK
1583,2025-26,2025-12-10,OKC,PHX
1584,2025-26,2025-12-10,LAL,SAS


In [427]:
final_data = merge_test[
    merge_test["_merge"] == "both"
].copy()

final_data = final_data.drop(columns=["_merge"])

print("Final matched games:", len(final_data))
print("Final shape:", final_data.shape)
print(final_data["SEASON"].value_counts())

Final matched games: 2442
Final shape: (2442, 87)
SEASON
2024-25    1221
2025-26    1221
Name: count, dtype: int64


In [428]:
stats = [
    "PTS",
    "REB",
    "AST",
    "FG_PCT",
    "FG3_PCT",
    "TOV",
    "PLUS_MINUS",
    "WIN"
]

feature_cols = []

for side in ["HOME", "AWAY"]:
    for stat in stats:
        feature_cols.append(f"{side}_{stat}_LAST5")
        feature_cols.append(f"{side}_{stat}_LAST10")

print("Number of model features:", len(feature_cols))
print(feature_cols)

Number of model features: 32
['HOME_PTS_LAST5', 'HOME_PTS_LAST10', 'HOME_REB_LAST5', 'HOME_REB_LAST10', 'HOME_AST_LAST5', 'HOME_AST_LAST10', 'HOME_FG_PCT_LAST5', 'HOME_FG_PCT_LAST10', 'HOME_FG3_PCT_LAST5', 'HOME_FG3_PCT_LAST10', 'HOME_TOV_LAST5', 'HOME_TOV_LAST10', 'HOME_PLUS_MINUS_LAST5', 'HOME_PLUS_MINUS_LAST10', 'HOME_WIN_LAST5', 'HOME_WIN_LAST10', 'AWAY_PTS_LAST5', 'AWAY_PTS_LAST10', 'AWAY_REB_LAST5', 'AWAY_REB_LAST10', 'AWAY_AST_LAST5', 'AWAY_AST_LAST10', 'AWAY_FG_PCT_LAST5', 'AWAY_FG_PCT_LAST10', 'AWAY_FG3_PCT_LAST5', 'AWAY_FG3_PCT_LAST10', 'AWAY_TOV_LAST5', 'AWAY_TOV_LAST10', 'AWAY_PLUS_MINUS_LAST5', 'AWAY_PLUS_MINUS_LAST10', 'AWAY_WIN_LAST5', 'AWAY_WIN_LAST10']


In [429]:
missing_features = [
    col for col in feature_cols
    if col not in final_data.columns
]

print("Missing features:", missing_features)

Missing features: []


In [430]:
# Create betting target: whether the home team covered the closing spread
import numpy as np
final_data["ATS_MARGIN"] = (
    final_data["PTS_HOME"]
    - final_data["PTS_AWAY"]
    + final_data["close_spread_home"]
)

final_data["HOME_COVER"] = np.where(
    final_data["ATS_MARGIN"] > 0,
    1,
    np.where(
        final_data["ATS_MARGIN"] < 0,
        0,
        np.nan
    )
)

print(final_data["HOME_COVER"].value_counts(dropna=False))

print(final_data[
    [
        "GAME_DATE",
        "HOME_TEAM",
        "AWAY_TEAM",
        "PTS_HOME",
        "PTS_AWAY",
        "close_spread_home",
        "ATS_MARGIN",
        "HOME_COVER"
    ]
].head(10))

HOME_COVER
1.0    1241
0.0    1200
NaN       1
Name: count, dtype: int64
   GAME_DATE HOME_TEAM AWAY_TEAM  PTS_HOME  PTS_AWAY  close_spread_home  \
0 2024-10-22       BOS       NYK       132       109               -6.5   
1 2024-10-22       LAL       MIN       110       103                1.5   
2 2024-10-23       DET       IND       109       115                5.5   
3 2024-10-23       ATL       BKN       120       116               -6.5   
4 2024-10-23       MIA       ORL        97       116               -2.5   
5 2024-10-23       PHI       MIL       109       124                4.0   
6 2024-10-23       TOR       CLE       106       136                6.5   
7 2024-10-23       HOU       CHA       105       110               -8.5   
8 2024-10-23       NOP       CHI       123       111               -4.5   
9 2024-10-23       UTA       MEM       124       126                2.5   

   ATS_MARGIN  HOME_COVER  
0        16.5         1.0  
1         8.5         1.0  
2        -0.5    

In [431]:
market_cols = [
    "close_spread_home",
    "HOME_COVER"
]

basic_cols = [
    "SEASON",
    "GAME_ID",
    "GAME_DATE",
    "HOME_TEAM",
    "AWAY_TEAM",
    "HOME_WIN"
]

processed_data = final_data[
    basic_cols + feature_cols + market_cols
].copy()

print("Processed data shape:", processed_data.shape)
print(processed_data.columns.tolist())


Processed data shape: (2442, 40)
['SEASON', 'GAME_ID', 'GAME_DATE', 'HOME_TEAM', 'AWAY_TEAM', 'HOME_WIN', 'HOME_PTS_LAST5', 'HOME_PTS_LAST10', 'HOME_REB_LAST5', 'HOME_REB_LAST10', 'HOME_AST_LAST5', 'HOME_AST_LAST10', 'HOME_FG_PCT_LAST5', 'HOME_FG_PCT_LAST10', 'HOME_FG3_PCT_LAST5', 'HOME_FG3_PCT_LAST10', 'HOME_TOV_LAST5', 'HOME_TOV_LAST10', 'HOME_PLUS_MINUS_LAST5', 'HOME_PLUS_MINUS_LAST10', 'HOME_WIN_LAST5', 'HOME_WIN_LAST10', 'AWAY_PTS_LAST5', 'AWAY_PTS_LAST10', 'AWAY_REB_LAST5', 'AWAY_REB_LAST10', 'AWAY_AST_LAST5', 'AWAY_AST_LAST10', 'AWAY_FG_PCT_LAST5', 'AWAY_FG_PCT_LAST10', 'AWAY_FG3_PCT_LAST5', 'AWAY_FG3_PCT_LAST10', 'AWAY_TOV_LAST5', 'AWAY_TOV_LAST10', 'AWAY_PLUS_MINUS_LAST5', 'AWAY_PLUS_MINUS_LAST10', 'AWAY_WIN_LAST5', 'AWAY_WIN_LAST10', 'close_spread_home', 'HOME_COVER']


In [432]:
nan_counts = processed_data.isna().sum()

print(nan_counts[nan_counts > 0])

HOME_PTS_LAST5            150
HOME_PTS_LAST10           299
HOME_REB_LAST5            150
HOME_REB_LAST10           299
HOME_AST_LAST5            150
HOME_AST_LAST10           299
HOME_FG_PCT_LAST5         150
HOME_FG_PCT_LAST10        299
HOME_FG3_PCT_LAST5        150
HOME_FG3_PCT_LAST10       299
HOME_TOV_LAST5            150
HOME_TOV_LAST10           299
HOME_PLUS_MINUS_LAST5     150
HOME_PLUS_MINUS_LAST10    299
HOME_WIN_LAST5            150
HOME_WIN_LAST10           299
AWAY_PTS_LAST5            150
AWAY_PTS_LAST10           301
AWAY_REB_LAST5            150
AWAY_REB_LAST10           301
AWAY_AST_LAST5            150
AWAY_AST_LAST10           301
AWAY_FG_PCT_LAST5         150
AWAY_FG_PCT_LAST10        301
AWAY_FG3_PCT_LAST5        150
AWAY_FG3_PCT_LAST10       301
AWAY_TOV_LAST5            150
AWAY_TOV_LAST10           301
AWAY_PLUS_MINUS_LAST5     150
AWAY_PLUS_MINUS_LAST10    301
AWAY_WIN_LAST5            150
AWAY_WIN_LAST10           301
HOME_COVER                  1
dtype: int

In [433]:
required_last10 = [
    col for col in processed_data.columns
    if "LAST10" in col
]

processed_data_clean = processed_data.dropna(
    subset=required_last10
).copy()

processed_data_clean = processed_data_clean.reset_index(drop=True)

print("Before:", len(processed_data))
print("After:", len(processed_data_clean))
print("Removed:", len(processed_data) - len(processed_data_clean))

rolling_cols = [
    col for col in processed_data_clean.columns
    if "LAST5" in col or "LAST10" in col
]

print(
    processed_data_clean[rolling_cols]
    .isna()
    .sum()
    .sum()
)


Before: 2442
After: 2130
Removed: 312
0


In [434]:
print(
    "Closing line NaN:",
    processed_data_clean["close_spread_home"].isna().sum()
)

print(
    processed_data_clean[
        ["GAME_DATE", "HOME_TEAM", "AWAY_TEAM", 
         "HOME_WIN", "close_spread_home"]
    ].head(10)
)

Closing line NaN: 0
   GAME_DATE HOME_TEAM AWAY_TEAM  HOME_WIN  close_spread_home
0 2024-11-10       POR       MEM         0                1.5
1 2024-11-11       CHI       CLE         0                8.5
2 2024-11-11       NOP       BKN         0                1.5
3 2024-11-11       OKC       LAC         1               -6.5
4 2024-11-11       SAS       SAC         1                2.5
5 2024-11-12       BOS       ATL         0              -16.5
6 2024-11-12       ORL       CHA         1               -8.5
7 2024-11-12       MIL       TOR         1               -8.5
8 2024-11-12       GSW       DAL         1               -2.5
9 2024-11-12       POR       MIN         1                9.5


In [435]:
push_games = final_data[final_data["ATS_MARGIN"] == 0]

print("Number of push games:", len(push_games))

print(
    push_games[
        [
            "GAME_DATE",
            "HOME_TEAM",
            "AWAY_TEAM",
            "PTS_HOME",
            "PTS_AWAY",
            "close_spread_home",
            "ATS_MARGIN"
        ]
    ]
)

Number of push games: 1
     GAME_DATE HOME_TEAM AWAY_TEAM  PTS_HOME  PTS_AWAY  close_spread_home  \
798 2025-02-12       OKC       MIA       115       101              -14.0   

     ATS_MARGIN  
798         0.0  


In [436]:
processed_data_clean.to_csv(
    "processed_data.csv",
    index=False
)

print("Saved processed_data.csv")
print("Final shape:", processed_data_clean.shape)
print(processed_data_clean["SEASON"].value_counts())

Saved processed_data.csv
Final shape: (2130, 40)
SEASON
2024-25    1065
2025-26    1065
Name: count, dtype: int64
